In [1]:
import sys
sys.path.append('third_party/Matcha-TTS')
sys.path.append('cdcd')

import math
import time

from IPython.display import Audio
from IPython.display import display

LETTERS = {"a": 0, "b": 1, "c": 2, "d": 3, "e": 4, "f": 5, "g": 6, "h": 7, "i": 8, "j": 9, 
           "k": 10, "l": 11, "m": 12, "n": 13, "o": 14, "p": 15, "q": 16, "r": 17, "s": 18, 
           "t": 19, "u": 20, "v": 21, "w": 22, "x": 23, "y": 24, "z": 25}
PUNCTUATION = {".": 26, ",": 27, "?": 28, "!": 29, ":": 30, ";": 31, "'": 32}
ALL_CHARS = {**LETTERS, **PUNCTUATION, " ": 33}

DURATIONS = {"a": 2.1, "b": 1.7, "c": 1.4, "d": 1.5, "e": 1.3, "f": 1.6, "g": 1.1, "h": 0.5, "i": 1.9, 
             "j": 2.7, "k": 1.1, "l": 2.2, "m": 1.8, "n": 1.8, "o": 1.6, "p": 1.4, "q": 1.3, "r": 1.8, 
             "s": 2.7, "t": 1.0, "u": 1.2, "v": 1.8, "w": 1.2, "x": 3.1, "y": 2.2, "z": 4.6, ".": 10.6, 
             ",": 4.7, "?": 13.7, "!": 15.3, ":": 13.9, ";": 8.2, "'": 0.1, " ": 1.6}

In [3]:
from cosyvoice.cli.cosyvoice import CosyVoice2
from cosyvoice.utils.file_utils import load_wav

from cdcd.diffusion import Diffusion
from cdcd.dit import DiT

import torch
import torchaudio

In [24]:
# load CosyVoice2 checkpoint
cosyvoice = CosyVoice2("../modules")

In [6]:
# load base diffusion checkpoint
dit = DiT(vocab_size=len(ALL_CHARS))
diffusion = Diffusion(dit)
diffusion.init_ema_model()
checkpoint = torch.load("../ckpt/cdcd.pt", weights_only=True, map_location="cpu")
diffusion.load_state_dict(checkpoint)
diffusion = diffusion.cuda()

In [26]:
# define speech prompt for zero-shot TTS
wav_path = "../prompts/en_prompt_4.wav"
ref_speech = load_wav(wav_path, 16000)

#ref_text = "must a name mean something?"
#ref_text = "she is now choosing skirt to wear."
#ref_text = "i do not eat bread."
ref_text = "i did go and made many prisoners."

print("Speech prompt:")
display(Audio(wav_path))
print("Text prompt: %s" % ref_text)

In [20]:
# text to synthesize
#text = "real time speech interaction is particularly valuable in scenarios requiring rapid feedback and immediate responses."
#text = "prior to the reformation, shared religion partially compensated for weak imperial institutions."
#text = "he is about three feet high, and is dressed in a little red jacket or roundabout, with red breeches buckled at the knee, gray or black stockings, and a hat cocked in the style of a century ago, over a little old withered face."
text = "she was born during the hundred years' war between england and france, which had begun over the status of english territories in france and english claims to the french throne."

In [27]:
# apply frontend
save_path = "../results/gen_4.wav"

# concatenate reference and synthesized text
text_concat = ref_text + " " + text
text_concat = text_concat.replace('"', "").replace("-", " ")
words_concat = [w.strip() for w in text_concat.split(" ") if len(w.strip()) > 0]
text_concat = " ".join(words_concat)
text_tokens = torch.LongTensor([ALL_CHARS[c] for c in text_concat if c in ALL_CHARS]).cuda()

# apply frontend
prompt_token, prompt_feat, embedding = cosyvoice.apply_frontend(text, ref_text, ref_speech)

# predict duration
chars_text = [c for c in text if c in ALL_CHARS]
chars_prompt = [c for c in ref_text if c in ALL_CHARS]
durs_text = [DURATIONS[c] for c in chars_text]
durs_prompt = [DURATIONS[c] for c in chars_prompt]
if prompt_token.shape[1] / sum(durs_prompt) > 1.1:
    dur_pred = math.ceil(sum(durs_text) * 1.1)
elif prompt_token.shape[1] / sum(durs_prompt) < 0.8:
    dur_pred = math.ceil(sum(durs_text) * 0.8)
else:
    dur_pred = math.ceil(sum(durs_text) * (prompt_token.shape[1] / sum(durs_prompt)))

# generate tokens with DM
dm_tokens = diffusion(seq_length=dur_pred, ref_tokens=prompt_token[0], text=text_tokens, w=0.5, 
                      prior_stddev=0.666, temperature=1.0, use_ema=True, n_timesteps=25)

# convert these tokens to waveform
result = cosyvoice.tokens_to_speech(dm_tokens, prompt_token, prompt_feat, embedding)

# play generated audio
torchaudio.save(save_path, result, cosyvoice.sample_rate)
display(Audio(save_path))